<a href="https://colab.research.google.com/github/SubhashMurmu/English-Kumaoni-Translator-BTP/blob/main/english_kumaoni_mbart_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece

In [ ]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
DATASET_NAME = "sourabsb/english-kumaoni_translation_dataset"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "hi_IN"

model = MBartForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)

dataset = load_dataset(DATASET_NAME, split="train")

def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["kumaoni"]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_weights",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    num_train_epochs=3,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("./lora_weights")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

kumaoni_dataset_final.jsonl:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11046 [00:00<?, ? examples/s]

Map:   0%|          | 0/11046 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,3.844049,3.747976
2,3.233640,3.252454
3,2.986061,3.083485


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("lora_weights", 'zip', "./lora_weights")
files.download("lora_weights.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>